## code to translate German to English and vice versa with deepL

In [1]:
import pandas as pd
from pprint import pprint
import os
import ast
import deepl
from tqdm import tqdm
import json
import requests
import matplotlib.pyplot as plt

In [2]:
# get connection to deepl api
deepl_key = open("/Users/sofiaromero/Documents/DeepL.txt").read()
deepl_client = deepl.DeepLClient(deepl_key)

In [3]:
# api functions 

def generate_glossaries(dict_path_l, s_lang, t_lang, auth_key):
    url = "https://api-free.deepl.com/v3/glossaries"
    header= {
        "Authorization": f"DeepL-Auth-Key {auth_key}",
        "User-Agent": "YourApp/1.2.3",
        "Content-Type": "application/json"
    }    
    glossaries = []
    for i,d in enumerate(dict_path_l):
        payload = {
            "name": f"glossary{i}",
            "dictionaries": [{
                # TODO: get languages from file name
                "source_lang": s_lang,
                "target_lang": t_lang,
                "entries": open(d,"r").read(),
                "entries_format": "csv"}
            ]}
        resp = requests.post(url, headers=header, json=payload)
        print(resp.status_code, resp.json())
        glossaries.append(resp.json())
    return glossaries


def get_glossary_ids(auth_key):
    url = "https://api-free.deepl.com/v3/glossaries"
    headers = {
        "Authorization": f"DeepL-Auth-Key {auth_key}",
        "User-Agent": "YourApp/1.2.3",
    }
    resp = requests.get(url, headers=headers, timeout=30)
    print(resp.status_code)
    return [x.get("glossary_id") for x in resp.json()["glossaries"]]


def transalate_to(text_l, s_lang, t_lang, glossary_id):
    res = []
    for text in text_l:
        if glossary_id:
            res.append(deepl_client.translate_text(text, target_lang=t_lang, source_lang=s_lang, glossary=glossary_id).text)
        else: 
            res.append(deepl_client.translate_text(text, target_lang=t_lang, source_lang=s_lang).text)
    return res


In [4]:
#generate_glossaries(["config_files/ger_to_eng.csv"], "DE", "EN", deepl_key)
# get all current glossaries
glossary_ids = get_glossary_ids(deepl_key)

200


In [5]:
# translate ki terms as test
'''
test_terms = list(ast.literal_eval(open("config_files/ger_anthroscore_KI_terms.txt").read()).values())
print(test_terms)
translations = transalate_to(test_terms, "DE", "EN-GB", glossary_ids[0])
print(translations)
'''

'\ntest_terms = list(ast.literal_eval(open("config_files/ger_anthroscore_KI_terms.txt").read()).values())\nprint(test_terms)\ntranslations = transalate_to(test_terms, "DE", "EN-GB", glossary_ids[0])\nprint(translations)\n'

In [6]:
# write text (from file) and its transaltion to df and save to csv and json file
def tranlate_to_df(txts_json, file_to):
    # load json as dict
    with open(txts_json, "r") as f:
        txts = json.load(f)

    res_df = pd.DataFrame([(key, var) for (key, L) in txts.items() for var in L], columns=['id', 'text']).drop_duplicates(subset=['text']).reset_index(drop=True)
    #fill df with traslations
    res_df['translation'] = transalate_to(res_df['text'].tolist(), "DE", "EN-GB", glossary_ids[0]) # TODO: changing source and target langs dependeing on input
    #save to file 
    res_df.to_csv(f"{file_to}.csv", index=False)
    res_df.to_json(f"{file_to}.json", orient="records")
    return res_df


### Translate KI_sentences to English

In [7]:
tranlate_to_df("generated_data/KI_sentences.json", "generated_data/KI_sentences_translated")

,id,text,translation
0,5315432.txt,der elek­tro­mo­tor pusht den rol­ler auf bis ...,the electric motor pushes the scooter up to 25...
1,5124643.txt,ungeneralsekretär ban ki moon bezeichnete die ...,secretary-general ban ki moon calls the killin...
2,5094180.txt,gegen dieses manko hilft auch die nominierung ...,"even the nomination of the chinese ""architect""..."
3,851887.txt,taz herr palm man liest von gewalt an ihrer sc...,taz mr palm you read about violence at your sc...
4,851887.txt,philipp palm es ist nicht so dass wir in einer...,philipp palm it is not that we are in a seriou...
...,...,...,...
1485,5570765.txt,der truck soll während der proteste abgesägte ...,the truck is said to have removed sawn-off log...
1486,5570765.txt,auf facebook postet er das offizielle dokument...,on facebook he posts the official document in ...
1487,5454189.txt,mit lina larissa strahl lisamarie koroll 1540 ki,with lina larissa strahl lisamarie koroll 1540 ki
1488,5823944.txt,einer dieser tests entwickelt von der philosop...,"one of these tests, developed by philosopher s..."


In [8]:
# input to anthroscore
# get AI-related words, sorted according to 'Häufigkeitsklassen' and from AnthroScore
# https://corpora.uni-leipzig.de/de/res?corpusId=deu_news_2024&word=KI
ki_terms = list(ast.literal_eval(open("config_files/ger_anthroscore_KI_terms.txt").read()).values())
ki_terms += ['KI','Künstliche Intelligenz', 'ChatGPT', 'AI', 'Copilot', 'Gemini', 'Chatbot','LLM', 'Chat-GPT',
            'Sprachmodell', 'KI-Modell', 'KI-System', 'KI-Tool', 'Artificial Intelligence', 'KI-Anwendung']
ai_terms = transalate_to(ki_terms, "DE", "EN-GB", glossary_ids[0])

In [9]:
with open(f"config_files/ai_terms.txt", "w") as f:
        f.writelines([" "+f"'{x}'" for x in ai_terms])
        f.close()

with open(f"config_files/ki_terms.txt", "w") as f:
        f.writelines([" "+f"'{x}'" for x in ki_terms])
        f.close()